# RL Interceptor Drone — Results

Visualizations of training and evaluation results.

1. **Intercept evaluation benchmark** — PPO v8.16 vs. classical baselines (`intercept/intercept_ppo_8.16.csv`)
2. *Training curves* — TODO
3. *Pursuit–evasion self-play* — TODO (`pursuit_evasion_selfplay/`)

## 0. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

VIS_DIR = Path.cwd()  # run from scripts/visualization

# Fixed method order and colors (RL first, so it always gets slot 1).
METHODS = ["rl", "pure_pursuit", "pn", "apf", "kinematic_mpc", "nonlinear_mpc"]
METHOD_LABELS = {
    "rl": "RL (PPO)",
    "pure_pursuit": "Pure pursuit",
    "pn": "Prop. navigation",
    "apf": "APF",
    "kinematic_mpc": "Kinematic MPC",
    "nonlinear_mpc": "Nonlinear MPC",
}
METHOD_COLORS = dict(zip(METHODS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]))
SCENARIOS = ["hover", "linear", "random"]

INK, INK_MUTED, GRID = "#1f1f1f", "#6b6b6b", "#e6e6e6"
plt.rcParams.update({
    "figure.dpi": 110,
    "font.size": 10,
    "text.color": INK,
    "axes.labelcolor": INK_MUTED,
    "axes.edgecolor": GRID,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.color": GRID,
    "axes.axisbelow": True,
    "xtick.color": INK_MUTED,
    "ytick.color": INK_MUTED,
    "legend.frameon": False,
})


def grouped_bars(ax, df, mean_col, std_col=None, scenarios=SCENARIOS, methods=METHODS, width=0.8):
    """One group per scenario, one bar per method; optional ±std error bars."""
    x = np.arange(len(scenarios))
    w = width / len(methods)
    for i, m in enumerate(methods):
        sub = df[df.method == m].set_index("scenario").reindex(scenarios)
        ax.bar(
            x - width / 2 + (i + 0.5) * w, sub[mean_col], w * 0.9,  # 10% gap between bars
            yerr=None if std_col is None else sub[std_col],
            color=METHOD_COLORS[m], label=METHOD_LABELS[m],
            error_kw=dict(ecolor=INK_MUTED, elinewidth=1, capsize=2),
        )
    ax.set_xticks(x, [s.capitalize() for s in scenarios])
    ax.tick_params(axis="x", length=0)
    return ax

## 1. Intercept evaluation benchmark — PPO v8.16

Each row is one (scenario, method) pair aggregated over `n` evaluation runs; `*_mean` / `*_std` are across runs.
Error bars below are ±1 std.

Scenarios: **hover** (stationary target), **linear** (constant-velocity target), **random** (randomly maneuvering target).

### 1.1 Load data

In [ ]:
bench = pd.read_csv(VIS_DIR / "intercept" / "intercept_ppo_8.16.csv")
bench["method"] = pd.Categorical(bench["method"], METHODS, ordered=True)
bench["scenario"] = pd.Categorical(bench["scenario"], SCENARIOS, ordered=True)
bench = bench.sort_values(["scenario", "method"]).reset_index(drop=True)
bench

### 1.2 Summary table (mean ± std)

In [ ]:
def pm(mean, std, fmt):
    return [f"{m:{fmt}} ± {s:{fmt}}" for m, s in zip(mean, std)]

summary = pd.DataFrame({
    "scenario": bench.scenario,
    "method": bench.method.map(METHOD_LABELS),
    "success rate [%]": pm(100 * bench.success_rate_mean, 100 * bench.success_rate_std, ".1f"),
    "intercept time [s]": pm(bench.intercept_time_s_mean, bench.intercept_time_s_std, ".2f"),
    "miss distance [m]": pm(bench.miss_distance_mean, bench.miss_distance_std, ".3f"),
    "avg travel vel [m/s]": bench.avg_travel_velocity.round(2),
    "avg intercept vel [m/s]": bench.avg_intercept_velocity.round(2),
}).set_index(["scenario", "method"])
summary

### 1.3 Success rate

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
plot_df = bench.assign(sr=100 * bench.success_rate_mean, sr_std=100 * bench.success_rate_std)
grouped_bars(ax, plot_df, "sr", "sr_std")
ax.set_ylabel("Success rate [%]")
ax.set_ylim(0, 105)
ax.set_title("Intercept success rate by scenario", loc="left")
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.08))
fig.tight_layout()
plt.show()

### 1.4 Intercept time and miss distance

Two separate panels (different units), each lower-is-better.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
grouped_bars(axes[0], bench, "intercept_time_s_mean", "intercept_time_s_std")
axes[0].set_ylabel("Intercept time [s]")
axes[0].set_title("Time to intercept (lower is better)", loc="left")
grouped_bars(axes[1], bench, "miss_distance_mean", "miss_distance_std")
axes[1].set_ylabel("Miss distance [m]")
axes[1].set_title("Miss distance (lower is better)", loc="left")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=6, loc="lower center", bbox_to_anchor=(0.5, -0.04))
fig.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()

### 1.5 Speed–success trade-off

Each point is one method; position shows success rate vs. intercept time (top-left is best).
One panel per scenario; methods are told apart by color and marker shape.

In [ ]:
MARKERS = dict(zip(METHODS, ["o", "s", "^", "D", "v", "P"]))
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True)
for ax, sc in zip(axes, SCENARIOS):
    sub = bench[bench.scenario == sc]
    for _, r in sub.iterrows():
        ax.errorbar(
            r.intercept_time_s_mean, 100 * r.success_rate_mean,
            xerr=r.intercept_time_s_std, yerr=100 * r.success_rate_std,
            fmt=MARKERS[r.method], ms=8, label=METHOD_LABELS[r.method], color=METHOD_COLORS[r.method], mec="white", mew=1.5,
            ecolor=INK_MUTED, elinewidth=1,
        )
    ax.set_title(sc.capitalize(), loc="left")
    ax.set_xlabel("Intercept time [s]")
    ax.grid(axis="x", color=GRID)
axes[0].set_ylabel("Success rate [%]")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=6, loc="lower center", bbox_to_anchor=(0.5, -0.04))
fig.tight_layout(rect=(0, 0.06, 1, 1))
plt.show()

### 1.6 Velocity profile

Average and peak speed during the approach (*travel*) and at the moment of intercept.
Max values are a single number across runs (no std).

In [ ]:
VEL_COLS = [
    ("avg_travel_velocity", "Avg travel speed [m/s]"),
    ("max_travel_velocity", "Max travel speed [m/s]"),
    ("avg_intercept_velocity", "Avg intercept speed [m/s]"),
    ("max_intercept_velocity", "Max intercept speed [m/s]"),
]
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
for ax, (col, label) in zip(axes.flat, VEL_COLS):
    grouped_bars(ax, bench, col)
    ax.set_title(label, loc="left")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=6, loc="lower center", bbox_to_anchor=(0.5, -0.02))
fig.tight_layout(rect=(0, 0.04, 1, 1))
plt.show()

### 1.7 RL margin over the best baseline

For each scenario: RL value minus the best classical baseline on that metric
(success: higher is better; time / miss distance: lower is better).

In [ ]:
rows = []
for sc in SCENARIOS:
    sub = bench[bench.scenario == sc].set_index("method")
    base = sub.drop(index="rl")
    rl = sub.loc["rl"]
    best_sr = base.success_rate_mean.idxmax()
    best_t = base.intercept_time_s_mean.idxmin()
    best_md = base.miss_distance_mean.idxmin()
    rows.append({
        "scenario": sc,
        "Δ success [pp]": 100 * (rl.success_rate_mean - base.loc[best_sr, "success_rate_mean"]),
        "best success baseline": METHOD_LABELS[best_sr],
        "Δ intercept time [s]": rl.intercept_time_s_mean - base.loc[best_t, "intercept_time_s_mean"],
        "best time baseline": METHOD_LABELS[best_t],
        "Δ miss distance [m]": rl.miss_distance_mean - base.loc[best_md, "miss_distance_mean"],
        "best miss baseline": METHOD_LABELS[best_md],
    })
pd.DataFrame(rows).set_index("scenario").round(3)

## 2. Training curves

TODO

## 3. Pursuit–evasion self-play

TODO